# 02 — Specialized Agents

*Level 6 — Multi-Agent RAG*

## Objective
Compare what each specialized agent actually produces for the same kind of task, on real data — and see the Research Agent combine two other agents' work into one broader pass.


In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
sys.path.insert(0, str(LEVEL_DIR))


In [2]:
from multiagent_common.dataset import prepare
from multiagent_common.retrieval import DenseRetriever
from multiagent_common.loader import load_agent_class

data = prepare()
corpus_texts = {cid: c["text"] for cid, c in data.corpus.items()}
retriever = DenseRetriever.from_corpus(corpus_texts)

RetrievalAgent = load_agent_class("retrieval-agent", "RetrievalAgent")
SqlAgent = load_agent_class("sql-agent", "SqlAgent")
GraphAgent = load_agent_class("graph-agent", "GraphAgent")
ResearchAgent = load_agent_class("research-agent", "ResearchAgent")

retrieval_agent = RetrievalAgent(retriever, data.corpus)
sql_agent = SqlAgent()


## Retrieval Agent vs. SQL Agent on their respective home turf


In [3]:
sample_q = list(data.questions.values())[0]
print("Question:", sample_q["question"])
print("Real answer:", sample_q["answer"])
result = retrieval_agent.run(sample_q["question"])
print("Retrieval Agent:", result.output[:200])


Question: How much were the company's debt obligations as of December 31, 2023?
Real answer: $2,299,887 thousand


Retrieval Agent: $2,299,887 thousand.


In [4]:
sql_result = sql_agent.run("How many distinct actors are in the database?")
print("SQL Agent:", sql_result.output)


SQL Agent: SQL: SELECT COUNT(DISTINCT actor_id) FROM actor LIMIT 1
Rows: [{'COUNT(DISTINCT actor_id)': 200}]


## Build a small company knowledge graph, then the Research Agent combining it with retrieval


In [5]:
import importlib.util

spec = importlib.util.spec_from_file_location("_graph_impl", LEVEL_DIR / "graph-agent" / "agent.py")
graph_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(graph_module)

sample_docs = list(data.corpus.values())[:8]
triples = [t for d in sample_docs for t in graph_module.extract_triples(d["text"])]
graph = graph_module.build_graph(triples)
graph_agent = graph_module.GraphAgent(graph)
print(f"Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")


Graph: 0 nodes, 0 edges


In [6]:
research_agent = ResearchAgent(retrieval_agent, graph_agent)
research_result = research_agent.run(sample_q["question"])
print("Research Agent (combined):", research_result.output[:300])
print(f"\nEvidence pooled: {len(research_result.evidence)} items "
      f"(doc: {len(result.evidence)}, graph: {len(graph_agent.run(sample_q['question']).evidence)})")


Research Agent (combined): The company's debt obligations as of December 31, 2023, totaled $2,299,887 thousand.

Here is a concise research summary combining both sources:

As of December 31, 2023, the company had total debt obligations of $2,299,887 thousand. In 2023, the company repaid debt of $1.2 billion consisting of flo

Evidence pooled: 5 items (doc: 5, graph: 0)


## What I observed

The Retrieval Agent got the exact real answer (`$2,299,887 thousand`, matching the dataset's ground truth exactly) from the filing excerpts alone.

The knowledge graph came back **empty (0 nodes, 0 edges)** on this run — entity extraction found nothing usable in these 8 sampled filing excerpts, likely because dense financial-statement text (numbers and line items) gives an LLM far less relational structure to extract than the narrative Wikipedia-style text Levels 3 and 5 used for their graphs. The Research Agent still produced a correct, complete answer anyway — it degraded gracefully to the document evidence alone rather than failing because one of its two internal sources had nothing to contribute.

`retrieval-agent` and `sql-agent` are narrow by design — each only ever touches one backend, which makes their failures easy to diagnose (a bad SQL answer is never the vector index's fault). `research-agent` deliberately gives that up for breadth: it calls both `retrieval-agent`'s and `graph-agent`'s underlying logic and asks the LLM to combine them — useful when a task genuinely needs both, and robust enough here to still work when one of them found nothing.

## Next

[03 — Parallel vs. Sequential Workflows](./03_parallel_vs_sequential_workflows.ipynb)
